<a href="https://colab.research.google.com/github/Bgail-ranu/Data-science/blob/main/Week6_Model_Evaluation_and_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Preparing Data**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Warehouse_cleaned.csv')
df.head()
df = df.drop(columns = ['MONTHE'])
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,YEAR,MONTH,SUPPLIER,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
0,2020,January,REPUBLIC NATIONAL DISTRIBUTING CO,WINE,0.00,0.0,2.0
1,2020,January,PWSWN INC,WINE,0.00,1.0,4.0
2,2020,January,RELIABLE CHURCHILL LLLP,BEER,0.00,0.0,1.0
3,2020,January,LANTERNA DISTRIBUTORS INC,WINE,0.00,0.0,1.0
4,2020,January,DIONYSOS IMPORTS INC,WINE,0.82,0.0,0.0


In [3]:
y = df['WAREHOUSE SALES']
x = df.drop(columns=['WAREHOUSE SALES'])

In [4]:
cat_features = x.select_dtypes(include=['object']).columns
num_features = x.select_dtypes(include=['int64', 'float64']).columns
cat_features, num_features

(Index(['MONTH', 'SUPPLIER', 'ITEM TYPE'], dtype='object'),
 Index(['YEAR', 'RETAIL SALES', 'RETAIL TRANSFERS'], dtype='object'))

In [5]:
scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown='ignore')
preprocessor = ColumnTransformer(
    transformers = [
        ('num', scaler, num_features),
        ('cat', encoder, cat_features)
    ]
)


In [6]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state = 42)

# **Comparing Models**

In [7]:
models = {
    "Linear Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]),

    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("dt_regressor", DecisionTreeRegressor(random_state = 42))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("rf_regressor", RandomForestRegressor(random_state = 42))
    ])
}

In [8]:
def evaluate_model(model, x_train, x_test, y_train, y_test):
  """
  Evaluate the model on the training and test dataset

  Return:
      The performance of the model: RMSE, MAE, R2 Score

  """

  #train the model
  model.fit(x_train, y_train)

  #make predictions
  y_preds = model.predict(x_test)

  #evalute performance
  mae = mean_absolute_error(y_test, y_preds)
  rmse = np.sqrt(mean_squared_error(y_test, y_preds))
  r2 = r2_score(y_test, y_preds)

  return mae, rmse, r2

In [ ]:

#training each model

results = []  #used to store the model's result

for model_name, model in models.items():
  #using the function to evaluate each model in the dictionary
  mae, rmse, r2 = evaluate_model(model, x_train, x_test, y_train, y_test)

  #getting the results
  results.append({
      "Model": model_name,
      "MAE": mae,
      "RMSE": rmse,
      "R2_Score": r2
  })

#creating a dataframe for the results

results_df = pd.DataFrame(results)
results_df

In [33]:
dt = Pipeline([
    ("preprocessor", preprocessor),
    ("decisiontree_regressor", DecisionTreeRegressor(random_state = 42))
])
mae, rmse, r2 = evaluate_model(dt, x_train, x_test, y_train, y_test)
print(f'MAE: {mae}')
print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

MAE: 36.30964668149683
RMSE: 258.8170185148705
R2 Score: 0.4700711851866334


In [38]:
rf = Pipeline([
    ("preprocessor", preprocessor),
    ("randomforest_regressor", RandomForestRegressor(random_state = 42))
])
mae, rmse, r2 = evaluate_model(rf, x_train, x_test, y_train, y_test)
print(f'MAE: {mae}')
print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

KeyboardInterrupt: 

In [13]:
xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("xgb_regressor", XGBRegressor(random_state = 42))
])
mae, rmse, r2 = evaluate_model(xgb, x_train, x_test, y_train, y_test)
print(f'MAE: {mae}')
print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

MAE: 31.880857445854605
RMSE: 213.06485309057436
R2 Score: 0.6408668416999399


In [39]:
df.shape

(198627, 7)

# **Cross Validation**

In [14]:
cv_scores = cross_val_score(models['Random Forest'],
                            x,
                            y,
                            scoring = 'neg_root_mean_squared_error', cv=5)
print(f'Average CV RMSE: {cv_scores.mean()}')

KeyboardInterrupt: 

In [ ]:
model_rf = models["Random Forest"]
model_rf.fit(x_train, y_train)

feature_names = model_rf.named_steps['preprocessor'].get_feature_names_out()
importances = model_rf.named_steps["rf_regressor"].feature_importances_

Importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by = "importance", ascending = False)

Importance_df.head(10)

In [10]:
param_grid = {
    "rf_regressor__n_estimators": [50, 100],
    "rf_regressor__max_depth": [None, 10, 20],
    "rf_regressor__min_samples_split": [2, 5]
}


#grid
grid = GridSearchCV(
    models["Random Forest"],
    param_grid = param_grid,
    scoring = "neg_root_mean_squared_error",
    cv = 3,
    n_jobs = -1
)

#fit to our training dataset
grid.fit(x_train, y_train)

KeyboardInterrupt: 